
# 1 - Imports and defining functions

In [1]:
import sys
print(sys.executable)

/data/home/user/anaconda3/envs/btp2/bin/python


In [2]:
import numpy as np
import torch
from pyFM.mesh import TriMesh

import meshplot as mp

def plot_mesh(myMesh,cmap=None):
    mp.plot(myMesh.vertlist, myMesh.facelist,c=cmap)
    
def double_plot(myMesh1,myMesh2,cmap1=None,cmap2=None):
    d = mp.subplot(myMesh1.vertlist, myMesh1.facelist, c=cmap1, s=[2, 2, 0])
    mp.subplot(myMesh2.vertlist, myMesh2.facelist, c=cmap2, s=[2, 2, 1], data=d)


def visu(vertices):
    min_coord,max_coord = np.min(vertices,axis=0,keepdims=True),np.max(vertices,axis=0,keepdims=True)
    cmap = (vertices-min_coord)/(max_coord-min_coord)
    return cmap

path = './input/faust_off/'
mesh1_name = 'tr_reg_076'
mesh2_name = 'tr_reg_079'
path1 = path + mesh1_name + '.off'
path2 = path + mesh2_name + '.off'

# 2- Loading meshes to match

In [3]:
mesh1 = TriMesh(path1, center=True, area_normalize=True)
mesh2 = TriMesh(path2, center=True, area_normalize=True)
vertices1 = mesh1.vertices  
vertices2 = mesh2.vertices 
faces1 = mesh1.faces 
faces2 = mesh2.faces 
double_plot(mesh1,mesh2,cmap1=None,cmap2=None)

## Correspondence for Unsupervised point cloud shape matching 

In [4]:
import numpy as np
import torch
from pyFM.mesh import TriMesh
import meshplot as mp

def plot_mesh(myMesh, cmap=None):
    mp.plot(myMesh.vertlist, myMesh.facelist, c=cmap)
    
def double_plot(myMesh1, myMesh2, cmap1=None, cmap2=None):
    d = mp.subplot(myMesh1.vertlist, myMesh1.facelist, c=cmap1, s=[2, 2, 0])
    mp.subplot(myMesh2.vertlist, myMesh2.facelist, c=cmap2, s=[2, 2, 1], data=d)

def visu(vertices):
    min_coord, max_coord = np.min(vertices, axis=0, keepdims=True), np.max(vertices, axis=0, keepdims=True)
    cmap = (vertices - min_coord) / (max_coord - min_coord)
    return cmap

path = './input/faust_off/'

mesh1_name = 'tr_reg_076'  
mesh2_name = 'tr_reg_079'  

path1 = path + mesh1_name + '.off'
path2 = path + mesh2_name + '.off'

mesh1 = TriMesh(path1, center=True, area_normalize=True) 
mesh2 = TriMesh(path2, center=True, area_normalize=True)  #
p2p_path = "./p2p_results_st_te/p2p_tr_reg_076_to_tr_reg_079_sinkhorn.txt"

p2p_pairs = np.loadtxt(p2p_path, dtype=int)

src_idx = p2p_pairs[:, 0] 
tgt_idx = p2p_pairs[:, 1]  
tgt_idx = np.clip(tgt_idx, 0, mesh2.vertlist.shape[0] - 1)
cmap1 = visu(mesh1.vertlist) 
cmap2 = cmap1[src_idx]         
double_plot(mesh1, mesh2, cmap1=cmap1, cmap2=cmap2)

In [12]:
import numpy as np
import torch
from pyFM.mesh import TriMesh
import meshplot as mp

def plot_mesh(myMesh, cmap=None):
    mp.plot(myMesh.vertlist, myMesh.facelist, c=cmap)
    
def double_plot(myMesh1, myMesh2, cmap1=None, cmap2=None):
    d = mp.subplot(myMesh1.vertlist, myMesh1.facelist, c=cmap1, s=[2, 2, 0])
    mp.subplot(myMesh2.vertlist, myMesh2.facelist, c=cmap2, s=[2, 2, 1], data=d)

def visu(vertices):
    min_coord, max_coord = np.min(vertices, axis=0, keepdims=True), np.max(vertices, axis=0, keepdims=True)
    cmap = (vertices - min_coord) / (max_coord - min_coord)
    return cmap

path = './input/faust_off/'

# IMPORTANT: Match the order in your P2P filename
mesh1_name = 'tr_reg_076'  # Source (shape2 in your P2P script)
mesh2_name = 'tr_reg_079'  # Target (shape1 in your P2P script)

path1 = path + mesh1_name + '.off'
path2 = path + mesh2_name + '.off'

mesh1 = TriMesh(path1, center=True, area_normalize=True)  # 079 (source)
mesh2 = TriMesh(path2, center=True, area_normalize=True)  # 060 (target)

# --------------------------------------------------
# Load p2p file: 079 → 060
# --------------------------------------------------
p2p_path = "./p2p_results_st_te/p2p_tr_reg_076_to_tr_reg_079_sinkhorn.txt"

# File format: [i_shape2(079), j_shape1(060)]
p2p_pairs = np.loadtxt(p2p_path, dtype=int)

src_idx = p2p_pairs[:, 0]  # Indices on 079
tgt_idx = p2p_pairs[:, 1]  # Mapped indices on 060

# Safety clamp
tgt_idx = np.clip(tgt_idx, 0, mesh2.vertlist.shape[0] - 1)

# --------------------------------------------------
# Colors: Transfer from SOURCE (079) to TARGET (060)
# --------------------------------------------------
cmap1 = visu(mesh1.vertlist)  # Colors on mesh1 (079)
cmap2 = cmap1[src_idx]         # Transfer colors from 079 to 060 using mapping

# --------------------------------------------------
# Plot
# --------------------------------------------------
double_plot(mesh1, mesh2, cmap1=cmap1, cmap2=cmap2)

In [13]:

import numpy as np

p2p_path = (
    './p2p_results_st_te/p2p_tr_reg_076_to_tr_reg_079_sinkhorn.txt'
)

p2p = np.loadtxt(p2p_path, dtype=int).flatten()

# convert 1-based indexing → 0-based if needed
if p2p.min() == 1:
    p2p -= 1

p2p = np.clip(p2p, 0, len(vertices2) - 1)


In [19]:
import numpy as np

# Load only valid numeric lines, skipping any bad ones
txt_data = np.genfromtxt('./p2p_results_st_te/p2p_tr_reg_076_to_tr_reg_079_sinkhorn.txt', dtype=int, invalid_raise=False)

# Extract the 2nd column (index 1)
second_col = txt_data[:, 1]
second_col = second_col[:5000]
print(second_col)

[2588   72    7 ...  302 1676  302]


In [15]:
cmap1 = visu(mesh1.vertlist)
cmap2_wks = cmap1[p2p]
cmap2_nn = cmap1[p2p]

double_plot(mesh1,mesh2,cmap1=cmap1,cmap2=cmap2_wks)

Invalid color array given! Supported are numpy arrays. <class 'numpy.ndarray'>


In [16]:
cmap2_wks

array([[0.50079751, 0.95180366, 0.23357959],
       [0.42740541, 0.93449019, 0.251871  ],
       [0.51931077, 0.953094  , 0.21798681],
       ...,
       [0.69446309, 0.48988599, 0.05725014],
       [0.07670349, 0.0062374 , 0.32037487],
       [0.52906063, 0.65155239, 0.33817899]])

In [ ]:
# for the whole folder, run once, but if you want for one file run next cell, added by lubesh
import os
import numpy as np
import trimesh
import scipy.io as sio
import gdist

def compute_geodesic_matrix(vertices, faces):
    V = vertices.shape[0]
    dist_matrix = np.zeros((V, V), dtype=np.float32)

    for i in range(V):
        # compute geodesic distance from vertex i to all others
        dist = gdist.compute_gdist(
            vertices.astype(np.float64),
            faces.astype(np.int32),
            source_indices=np.array([i], dtype=np.int32)
        )
        dist_matrix[i] = dist

        if i % 500 == 0:
            print(f"Processed {i}/{V}")

    return dist_matrix


def process_faust(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)

    files = sorted([f for f in os.listdir(input_dir) if f.endswith('.off')])

    for f in files:
        mesh_path = os.path.join(input_dir, f)
        name = os.path.splitext(f)[0]

        print(f"\nProcessing {name}")

        mesh = trimesh.load(mesh_path, process=False)
        vertices = mesh.vertices
        faces = mesh.faces

        dist_matrix = compute_geodesic_matrix(vertices, faces)

        out_path = os.path.join(output_dir, name + ".mat")
        sio.savemat(out_path, {'dist': dist_matrix})

        print(f"Saved: {out_path}")


# ====== USAGE ======
input_dir = "./input/faust_off"      # folder with .off files
output_dir = "./input/faust_mat"    # where .mat will be saved

process_faust(input_dir, output_dir)


Processing tr_reg_000
Processed 0/4999
Processed 500/4999
Processed 1000/4999
Processed 1500/4999
Processed 2000/4999
Processed 2500/4999
Processed 3000/4999
Processed 3500/4999
Processed 4000/4999
Processed 4500/4999
Saved: ./input/faust_mat/tr_reg_000.mat

Processing tr_reg_001
Processed 0/5000
Processed 500/5000
Processed 1000/5000
Processed 1500/5000
Processed 2000/5000
Processed 2500/5000
Processed 3000/5000
Processed 3500/5000
Processed 4000/5000
Processed 4500/5000
Saved: ./input/faust_mat/tr_reg_001.mat

Processing tr_reg_002
Processed 0/4999
Processed 500/4999
Processed 1000/4999
Processed 1500/4999
Processed 2000/4999
Processed 2500/4999
Processed 3000/4999
Processed 3500/4999
Processed 4000/4999
Processed 4500/4999
Saved: ./input/faust_mat/tr_reg_002.mat

Processing tr_reg_003
Processed 0/5001
Processed 500/5001
Processed 1000/5001
Processed 1500/5001
Processed 2000/5001
Processed 2500/5001
Processed 3000/5001
Processed 3500/5001
Processed 4000/5001
Processed 4500/5001
Proc

In [ ]:
# addded by lubesh
import numpy as np
import trimesh
import scipy.io as sio
import gdist

def compute_geodesic_matrix(vertices, faces):
    V = vertices.shape[0]
    dist_matrix = np.zeros((V, V), dtype=np.float32)

    for i in range(V):
        dist = gdist.compute_gdist(
            vertices.astype(np.float64),
            faces.astype(np.int32),
            source_indices=np.array([i], dtype=np.int32)
        )
        dist_matrix[i] = dist

        if i % 500 == 0:
            print(f"Processed {i}/{V}")

    return dist_matrix


# ====== INPUT ======
mesh_path = "./input/faust_off/tr_reg_076.off"   # change this
output_path = "./input/faust_off/tr_reg_076.mat"

# ====== LOAD MESH ======
mesh = trimesh.load(mesh_path, process=False)
vertices = mesh.vertices
faces = mesh.faces

print(f"Vertices: {vertices.shape}, Faces: {faces.shape}")

# ====== COMPUTE ======
dist_matrix = compute_geodesic_matrix(vertices, faces)

# ====== SAVE ======
sio.savemat(output_path, {'dist': dist_matrix})

print(f"Saved to {output_path}")


# ====== INPUT ======
mesh_path = "./input/faust_off/tr_reg_079.off"   # change this
output_path = "./input/faust_mat/tr_reg_079.mat"

# ====== LOAD MESH ======
mesh = trimesh.load(mesh_path, process=False)
vertices = mesh.vertices
faces = mesh.faces

print(f"Vertices: {vertices.shape}, Faces: {faces.shape}")

# ====== COMPUTE ======
dist_matrix = compute_geodesic_matrix(vertices, faces)

# ====== SAVE ======
sio.savemat(output_path, {'dist': dist_matrix})

print(f"Saved to {output_path}")

Vertices: (5002, 3), Faces: (10000, 3)
Processed 0/5002
Processed 500/5002
Processed 1000/5002
Processed 1500/5002
Processed 2000/5002
Processed 2500/5002
Processed 3000/5002
Processed 3500/5002
Processed 4000/5002
Processed 4500/5002
Processed 5000/5002
Saved to ./input/faust_off/tr_reg_076.mat
Vertices: (4999, 3), Faces: (9994, 3)
Processed 0/4999
Processed 500/4999
Processed 1000/4999
Processed 1500/4999
Processed 2000/4999
Processed 2500/4999
Processed 3000/4999
Processed 3500/4999
Processed 4000/4999
Processed 4500/4999
Saved to ./input/faust_off/tr_reg_079.mat


In [ ]:
# 0.0	Perfect correspondence
# < 0.02	Excellent (SOTA level)
# 0.02 – 0.05	Very good
# 0.05 – 0.1	Good
# 0.1 – 0.2	Moderate
# > 0.2	Poor
# ≈ 1.0	Completely wrong mapping
import scipy.io as sio

def calculate_geodesic_error(dist_x, corr_x, corr_y, p2p, return_mean=True):
    """
    Calculate the geodesic error between predicted correspondence and gt correspondence

    Args:
        dist_x (np.ndarray): Geodesic distance matrix of shape x. shape [Vx, Vx]
        corr_x (np.ndarray): Ground truth correspondences of shape x. shape [V]
        corr_y (np.ndarray): Ground truth correspondences of shape y. shape [V]
        p2p (np.ndarray): Point-to-point map (shape y -> shape x). shape [Vy]
        return_mean (bool, optional): Average the geodesic error. Default True.
    Returns:
        avg_geodesic_error (np.ndarray): Average geodesic error.
    """
    ind21 = np.stack([corr_x, p2p[corr_y]], axis=-1)
    ind21 = np.ravel_multi_index(ind21.T, dims=[dist_x.shape[0], dist_x.shape[0]])
    geo_err = np.take(dist_x, ind21)
    if return_mean:
        return geo_err.mean()
    else:
        return geo_err

mat_path = './input/faust_mat/'
mat = sio.loadmat(mat_path + mesh1_name  + '.mat')
matrix = torch.from_numpy(mat['dist']).float()
corr_x = np.loadtxt('./input/FAUST/corres/' + mesh1_name + '.vts',dtype = np.int32)-1
corr_y = np.loadtxt('./input/FAUST/corres/' + mesh2_name + '.vts',dtype = np.int32)-1

error = calculate_geodesic_error(matrix,corr_x,corr_y,p2p).item()
error = error / matrix.max().item() # added by lubesh
print(f"{error:.3f}")

0.385
